# 04 — Subword-Label Alignment

**Goal:** Solve the last puzzle before fine-tuning NER. Our labels are at the **word** level (`APT28` → `B-THREAT_ACTOR`). But BERT tokenizes into **subwords** (`apt`, `##28`). How do labels map across?

This is the single most common source of bugs in BERT NER code. We'll do it carefully.

## What you'll learn

1. Why word-level labels can't be used directly
2. The `is_split_into_words=True` tokenizer flag
3. `word_ids()` — the mapping from each subword back to its source word
4. The standard convention: label the **first** subword, mark the rest with `-100` (ignored in loss)
5. A reusable `align_labels_with_tokens()` function you'll carry into later notebooks

## Step 1 — The problem, in one picture

Our word-level labels (from notebook 3):

```
APT28     -> B-THREAT_ACTOR
deployed  -> O
X-Agent   -> B-MALWARE
```

BERT's subword tokens:

```
[CLS] apt ##28 deployed x - agent [SEP]
```

3 words → 6 content tokens + 2 specials = 8 positions. Our label list has 3 entries. The lengths don't match, so we can't just zip them.

## Step 2 — `is_split_into_words=True`

When your input is already split into words (list of strings), tell the tokenizer so it can track which subword came from which word.

In [1]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

words = ["APT28", "deployed", "X-Agent", "malware", "exploiting", "CVE-2017-0199", "."]
labels = ["B-THREAT_ACTOR", "O", "B-MALWARE", "O", "O", "B-VULNERABILITY", "O"]

encoded = tokenizer(words, is_split_into_words=True, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])
print("Tokens:", tokens)

Tokens: ['[CLS]', 'apt', '##28', 'deployed', 'x', '-', 'agent', 'mal', '##ware', 'exploit', '##ing', 'cv', '##e', '-', '2017', '-', '01', '##9', '##9', '.', '[SEP]']


## Step 3 — `word_ids()`: the mapping

`encoded.word_ids()` returns a list the same length as the tokens, where each entry is the **index of the source word** (or `None` for special tokens).

In [2]:
word_ids = encoded.word_ids()

print(f"{'Pos':<4} {'Token':<14} {'word_id':<8} {'source word'}")
print("-" * 50)
for i, (tok, wid) in enumerate(zip(tokens, word_ids)):
    source = words[wid] if wid is not None else "(special)"
    print(f"{i:<4} {tok:<14} {str(wid):<8} {source}")

Pos  Token          word_id  source word
--------------------------------------------------
0    [CLS]          None     (special)
1    apt            0        APT28
2    ##28           0        APT28
3    deployed       1        deployed
4    x              2        X-Agent
5    -              2        X-Agent
6    agent          2        X-Agent
7    mal            3        malware
8    ##ware         3        malware
9    exploit        4        exploiting
10   ##ing          4        exploiting
11   cv             5        CVE-2017-0199
12   ##e            5        CVE-2017-0199
13   -              5        CVE-2017-0199
14   2017           5        CVE-2017-0199
15   -              5        CVE-2017-0199
16   01             5        CVE-2017-0199
17   ##9            5        CVE-2017-0199
18   ##9            5        CVE-2017-0199
19   .              6        .
20   [SEP]          None     (special)


Notice: `apt` and `##28` both have `word_id = 0` — they came from the same source word `APT28`.

## Step 4 — The labeling convention

The Hugging Face convention (from the official NER tutorial):

- **Special tokens** (`[CLS]`, `[SEP]`, padding): label `= -100`. PyTorch's cross-entropy loss skips `-100`.
- **First subword** of a word: use the word's label (e.g., `B-THREAT_ACTOR`).
- **Subsequent subwords** of the same word: label `= -100` (ignored in loss).

Alternative: replace `B-X` with `I-X` for continuation subwords. Both work; the `-100` approach is simpler and standard.

In [3]:
entity_types = ["THREAT_ACTOR", "MALWARE", "VULNERABILITY"]
label_list = ["O"] + [f"{p}-{t}" for t in entity_types for p in ("B", "I")]
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
print("label2id:", label2id)

label2id: {'O': 0, 'B-THREAT_ACTOR': 1, 'I-THREAT_ACTOR': 2, 'B-MALWARE': 3, 'I-MALWARE': 4, 'B-VULNERABILITY': 5, 'I-VULNERABILITY': 6}


In [4]:
def align_labels_with_tokens(word_labels, word_ids):
    """Turn word-level BIO labels into token-level label IDs.
    Special tokens and non-first subwords get -100 (ignored in loss)."""
    aligned = []
    previous_wid = None
    for wid in word_ids:
        if wid is None:
            aligned.append(-100)
        elif wid != previous_wid:
            aligned.append(label2id[word_labels[wid]])
        else:
            aligned.append(-100)
        previous_wid = wid
    return aligned


aligned = align_labels_with_tokens(labels, word_ids)

print(f"{'Pos':<4} {'Token':<14} {'Label ID':<10} {'Label name'}")
print("-" * 50)
for tok, lid in zip(tokens, aligned):
    name = id2label[lid] if lid != -100 else "(ignored)"
    print(f"{'':<4} {tok:<14} {lid:<10} {name}")

Pos  Token          Label ID   Label name
--------------------------------------------------
     [CLS]          -100       (ignored)
     apt            1          B-THREAT_ACTOR
     ##28           -100       (ignored)
     deployed       0          O
     x              3          B-MALWARE
     -              -100       (ignored)
     agent          -100       (ignored)
     mal            0          O
     ##ware         -100       (ignored)
     exploit        0          O
     ##ing          -100       (ignored)
     cv             5          B-VULNERABILITY
     ##e            -100       (ignored)
     -              -100       (ignored)
     2017           -100       (ignored)
     -              -100       (ignored)
     01             -100       (ignored)
     ##9            -100       (ignored)
     ##9            -100       (ignored)
     .              0          O
     [SEP]          -100       (ignored)


## Step 5 — Batch version (what you'll actually use in training)

Real training processes batches of examples. Here's the batched version — this exact function reappears in notebook 5.

In [5]:
def tokenize_and_align(examples, tokenizer, label2id):
    """examples: dict with keys 'words' (list of list of str) and 'labels' (list of list of str).
    Returns a tokenized batch with aligned 'labels' as ints (-100 for ignored)."""
    tokenized = tokenizer(
        examples["words"],
        is_split_into_words=True,
        truncation=True,
        padding=True,
    )
    all_labels = []
    for i, word_labels in enumerate(examples["labels"]):
        wids = tokenized.word_ids(batch_index=i)
        aligned = []
        previous = None
        for wid in wids:
            if wid is None:
                aligned.append(-100)
            elif wid != previous:
                aligned.append(label2id[word_labels[wid]])
            else:
                aligned.append(-100)
            previous = wid
        all_labels.append(aligned)
    tokenized["labels"] = all_labels
    return tokenized


# Toy 2-example batch
examples = {
    "words": [
        ["APT28", "deployed", "X-Agent", "."],
        ["The", "Lazarus", "Group", "used", "Cobalt", "Strike", "."],
    ],
    "labels": [
        ["B-THREAT_ACTOR", "O", "B-MALWARE", "O"],
        ["O", "B-THREAT_ACTOR", "I-THREAT_ACTOR", "O", "B-MALWARE", "I-MALWARE", "O"],
    ],
}

batch = tokenize_and_align(examples, tokenizer, label2id)

for i in range(len(examples["words"])):
    toks = tokenizer.convert_ids_to_tokens(batch["input_ids"][i])
    lbls = batch["labels"][i]
    print(f"--- Example {i} ---")
    for t, l in zip(toks, lbls):
        name = id2label[l] if l != -100 else "(ignored)"
        print(f"  {t:<14} {l:<5} {name}")
    print()

--- Example 0 ---
  [CLS]          -100  (ignored)
  apt            1     B-THREAT_ACTOR
  ##28           -100  (ignored)
  deployed       0     O
  x              3     B-MALWARE
  -              -100  (ignored)
  agent          -100  (ignored)
  .              0     O
  [SEP]          -100  (ignored)

--- Example 1 ---
  [CLS]          -100  (ignored)
  the            0     O
  lazarus        1     B-THREAT_ACTOR
  group          2     I-THREAT_ACTOR
  used           0     O
  cobalt         3     B-MALWARE
  strike         4     I-MALWARE
  .              0     O
  [SEP]          -100  (ignored)



## Your turn — exercises

1. Modify `align_labels_with_tokens` to use the **alternative convention**: replace `B-X` with `I-X` for continuation subwords instead of `-100`. Compare outputs.
2. What happens if you pass `truncation=True` with a sentence long enough to truncate? Add a long sentence and print the aligned labels — confirm the function still works and that `labels` matches `input_ids` in length.
3. Add `return_tensors="pt"` to the tokenizer call in `tokenize_and_align`. Does it still work? What changes in the output types?

## Summary

- Word-level labels can't directly feed BERT because of subword splitting.
- `is_split_into_words=True` + `.word_ids()` gives you the mapping.
- Convention: first subword keeps its word's label; rest get `-100` (ignored).
- `-100` is a sentinel PyTorch's `CrossEntropyLoss` skips.

## Next notebook

**`05_ner_finetuning_end_to_end.ipynb`** — put it all together. Load a small CTI-style dataset, apply `tokenize_and_align`, wire up `Trainer`, and fine-tune your first NER model.